# Compare signal-discovery results

Choose any two methods in the widgets to compare their output on the same conditions. MiniLM, full-list Decider, and Qwen3 reranker score the full fixed candidate list; Decider graph + history-conditioned reranking scores only its discovered shortlist. Rank comparisons use the final returned order; Jensen–Shannon divergence and shared weight mass are shown only for full-pool distributions. Graph-reranker probabilities are conditional on the discovered shortlist, not the full candidate pool.

In [8]:
import json
import math
import numpy as np
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import ipywidgets as widgets
from IPython.display import display

RESULT_FILES = {
    'MiniLM': Path('/storage/lcostes/ray-share/experiments/simbench-signals/minilm.jsonl'),
    'Decider': Path('/storage/lcostes/ray-share/experiments/simbench-signals/decider.jsonl'),
    'Qwen3 reranker': Path('/storage/lcostes/ray-share/experiments/simbench-signals/qwen3-reranker.jsonl'),
    'Decider graph + history rerank': Path('/storage/lcostes/ray-share/experiments/simbench-signals/decider-graph-reranked.jsonl'),
}

# Select any two distinct model names from RESULT_FILES, then rerun this and the comparison cells.
COMPARE_MODELS = ('Qwen3 reranker', 'Decider')
if len(COMPARE_MODELS) != 2 or COMPARE_MODELS[0] == COMPARE_MODELS[1]:
    raise ValueError('COMPARE_MODELS must contain two different model names')
unknown_models = set(COMPARE_MODELS) - set(RESULT_FILES)
if unknown_models:
    raise ValueError(f'Unknown model(s): {sorted(unknown_models)}')

def load_jsonl(path):
    with path.open(encoding='utf-8') as f:
        return [json.loads(line) for line in f if line.strip()]

RESULTS = {}
for name, path in RESULT_FILES.items():
    if path.is_file():
        RESULTS[name] = {row['condition_id']: row for row in load_jsonl(path)}
    else:
        print(f'Method output not yet available: {name}: {path}')
missing_selected = set(COMPARE_MODELS) - set(RESULTS)
if missing_selected:
    raise FileNotFoundError(f'Missing selected method output(s): {sorted(missing_selected)}')
CONDITION_IDS = sorted(set.intersection(*(set(RESULTS[name]) for name in COMPARE_MODELS)))
ALL_SHARED_CONDITION_IDS = sorted(set.intersection(*(set(RESULTS[name]) for name in RESULTS)))
if not CONDITION_IDS:
    raise ValueError(f'No shared conditions for {COMPARE_MODELS}')
print(f'Comparing {COMPARE_MODELS[0]} vs {COMPARE_MODELS[1]}; {len(CONDITION_IDS)} shared conditions.')

Method output not yet available: Decider graph + history rerank: /storage/lcostes/ray-share/experiments/simbench-signals/decider-graph-reranked.jsonl
Comparing Qwen3 reranker vs Decider; 5 shared conditions.


In [9]:
# Quick overview: condition wording and each selected method’s top-ranked signal.
def _top_signal_id(row):
    probabilities = row.get('choice_probabilities') or {}
    if probabilities:
        return max(probabilities, key=lambda key: (float(probabilities[key]), key))
    return max(row['signals'], key=lambda signal: float(signal.get('score', 0)))['id']

overview = []
for condition_id in CONDITION_IDS:
    row = RESULTS[COMPARE_MODELS[0]][condition_id]
    record = {'condition': condition_id, 'intent': row['condition']}
    for model in COMPARE_MODELS:
        record[f'{model} top signal'] = _top_signal_id(RESULTS[model][condition_id])
    overview.append(record)
display(pd.DataFrame(overview).set_index('condition'))

,intent,Qwen3 reranker top signal,Decider top signal
condition,,,
condition-001,I'm looking for products that serve similar pu...,attribute:customer:customer>review>product:pro...,attribute:review:review>product:product.descri...
condition-002,Which products appeal to similar customers bas...,attribute:customer:customer>review>product:pro...,relationship:product:product>review>customer
condition-003,Show me reviews that provide useful insights i...,attribute:product:product>review:review.review...,attribute:review:review:review.review_text
condition-004,Prioritize products that offer a good balance ...,attribute:customer:customer>review>product:pro...,attribute:review:review:review.rating
condition-005,I'd like to discover customers with similar ta...,attribute:customer:customer>review:review.rating,attribute:review:review:review.rating


In [10]:
def _ranked(row, method):
    probabilities = row.get('choice_probabilities') or {}
    if probabilities:
        ordered_ids = [signal_id for signal_id, _ in sorted(
            probabilities.items(), key=lambda item: (-float(item[1]), item[0])
        )]
    else:
        ordered_ids = [s['id'] for s in sorted(
            row['signals'], key=lambda s: float(s.get('score', 0)), reverse=True
        )]
    return pd.DataFrame({
        'signal_id': ordered_ids,
        f'{method} rank': range(1, len(ordered_ids) + 1),
    })

def _has_full_distribution(row):
    if str(row.get('method', '')).startswith('decider-graph'):
        return False
    if row.get('choice_probabilities'):
        return True
    signals = row.get('signals', [])
    return len(signals) >= int(row.get('candidate_count', len(signals)))

def _normalized_candidate_weights(model, row):
    if not _has_full_distribution(row):
        raise ValueError(f'{model} does not provide a full-candidate weight distribution')
    probabilities = row.get('choice_probabilities') or {}
    if probabilities:
        weights = {str(key): max(0.0, float(value)) for key, value in probabilities.items()}
    else:
        logits = {}
        for signal in row.get('signals', []):
            if 'ranking_logit' in signal:
                logit = float(signal['ranking_logit'])
            else:
                score = min(max(float(signal.get('score', 0.0)), 1e-12), 1 - 1e-12)
                logit = math.log(score) - math.log1p(-score)
            logits[str(signal['id'])] = logit
        if not logits:
            raise ValueError(f'No ranked candidates in {model} result')
        offset = max(logits.values())
        weights = {key: math.exp(value - offset) for key, value in logits.items()}
    total = sum(weights.values())
    if total <= 0:
        raise ValueError(f'{model} candidate weights have zero mass')
    return {key: value / total for key, value in weights.items()}


def _jensen_shannon_divergence(p, q):
    if set(p) != set(q):
        raise ValueError('Jensen-Shannon inputs must contain the same candidates')
    divergence = 0.0
    for key in p:
        midpoint = (p[key] + q[key]) / 2
        if p[key] > 0:
            divergence += 0.5 * p[key] * math.log2(p[key] / midpoint)
        if q[key] > 0:
            divergence += 0.5 * q[key] * math.log2(q[key] / midpoint)
    return divergence


def compare_condition(condition_id, top_n=15):
    if condition_id not in CONDITION_IDS:
        raise KeyError(f'No paired result for {condition_id} and {COMPARE_MODELS}')
    rows = {model: RESULTS[model][condition_id] for model in COMPARE_MODELS}
    ranked = {model: _ranked(rows[model], model) for model in COMPARE_MODELS}

    # Candidate descriptions/routes are shared metadata, not model-specific.
    catalog = {s['id']: s for row in rows.values() for s in row.get('signals', [])}
    all_signal_ids = set().union(*(set(frame['signal_id']) for frame in ranked.values()))
    comparison = pd.DataFrame([
        {
            'signal_id': signal_id,
            'kind': (s := catalog.get(signal_id, {})).get('kind'),
            'signal': (f"{s.get('attribute_table')}.{s.get('attribute_column')}"
                       if s.get('kind') == 'attribute' else 'relationship'),
            'route': ' → '.join(s.get('path') or []),
        }
        for signal_id in all_signal_ids
    ])
    for model, frame in ranked.items():
        comparison = comparison.merge(frame, on='signal_id', how='left')
    rank_columns = [f'{model} rank' for model in COMPARE_MODELS]
    comparison['best rank'] = comparison[rank_columns].min(axis=1)
    comparison = comparison.sort_values(['best rank', *rank_columns], na_position='last')
    print(rows[COMPARE_MODELS[0]]['condition'])
    print(f"Returned signals: {', '.join(f'{name}={len(ranked[name])}' for name in COMPARE_MODELS)}")
    print(f'Sorted by best (lowest) rank from {COMPARE_MODELS[0]} or {COMPARE_MODELS[1]}.')
    print('Top-k overlap (shared candidates among each method’s returned top k):')
    for k in (5, 10):
        a = set(ranked[COMPARE_MODELS[0]].head(k)['signal_id'])
        b = set(ranked[COMPARE_MODELS[1]].head(k)['signal_id'])
        denominator = min(k, len(a), len(b))
        print(f'  k={k}: {len(a & b)}/{denominator}')
    columns = ['kind', 'signal', 'route', *rank_columns]
    display(comparison[columns].head(top_n).style.format(
        {column: '{:.0f}' for column in rank_columns}, na_rep='—'
    ))

    paired = comparison.dropna(subset=rank_columns).copy()
    if paired.empty:
        print('No signals were returned by both methods, so a shared-rank plot is unavailable.')
        return
    full_distribution = all(_has_full_distribution(rows[name]) for name in COMPARE_MODELS)
    if full_distribution:
        model_a, model_b = COMPARE_MODELS
        weights_a = _normalized_candidate_weights(model_a, rows[model_a])
        weights_b = _normalized_candidate_weights(model_b, rows[model_b])
        shared_ids = sorted(set(weights_a) & set(weights_b))
        total_a = sum(weights_a[sid] for sid in shared_ids)
        total_b = sum(weights_b[sid] for sid in shared_ids)
        if total_a <= 0 or total_b <= 0:
            raise ValueError('Candidate weights must have positive total mass')
        p = {sid: weights_a[sid] / total_a for sid in shared_ids}
        q = {sid: weights_b[sid] / total_b for sid in shared_ids}
        js_divergence = _jensen_shannon_divergence(p, q)
        shared_mass = sum(min(p[sid], q[sid]) for sid in shared_ids)
        mean_weight = {sid: (p[sid] + q[sid]) / 2 for sid in shared_ids}
        print(f'Jensen–Shannon divergence: {js_divergence:.3f} bits (0 = identical, 1 = maximally different)')
        print(f'Shared normalized weight mass: {shared_mass:.3f} (1 = identical distributions)')
        point_weights = paired['signal_id'].map(mean_weight).tolist()
        max_weight = max(point_weights) if point_weights else 1.0
        point_sizes = [35 + 1100 * weight / max_weight for weight in point_weights]
    else:
        print('Distribution metrics are omitted: graph-search action probabilities are local to each decision, not a full-candidate distribution.')
        point_weights = None
        point_sizes = 55

    rank_a, rank_b = [f'{name} rank' for name in COMPARE_MODELS]
    limit = max(1, int(paired[[rank_a, rank_b]].max().max()))
    fig, ax = plt.subplots(figsize=(7, 6))
    if full_distribution:
        points = ax.scatter(
            paired[rank_a], paired[rank_b], c=point_weights, s=point_sizes,
            cmap='viridis', alpha=.85, edgecolors='black', linewidths=.25,
        )
        fig.colorbar(points, ax=ax, label='Mean normalized candidate weight')
    else:
        ax.scatter(
            paired[rank_a], paired[rank_b], color='#4C78A8', s=point_sizes,
            alpha=.85, edgecolors='black', linewidths=.25,
        )
    ax.plot([1, limit], [1, limit], '--', color='gray', linewidth=1)
    tick_step = max(1, limit // 8)
    rank_ticks = list(range(1, limit + 1, tick_step))
    if limit not in rank_ticks:
        rank_ticks.append(limit)
    plot_title = 'shared returned-signal ranks' if not full_distribution else 'full ranking comparison'
    ax.set(xlabel=f'{COMPARE_MODELS[0]} rank (1 = highest)',
           ylabel=f'{COMPARE_MODELS[1]} rank (1 = highest)',
           title=f'{condition_id}: {plot_title}',
           xlim=(0.5, limit + 0.5), ylim=(0.5, limit + 0.5),
           xticks=rank_ticks, yticks=rank_ticks)
    ax.grid(alpha=.2)
    fig.tight_layout()
    plt.show()


In [11]:
def browse_comparison(model_a, model_b, condition_id, top_n):
    global COMPARE_MODELS, CONDITION_IDS
    if model_a == model_b:
        print('Choose two different models.')
        return
    COMPARE_MODELS = (model_a, model_b)
    CONDITION_IDS = sorted(set(RESULTS[model_a]) & set(RESULTS[model_b]))
    if condition_id not in CONDITION_IDS:
        print(f'{condition_id} is not present for both selected models.')
        return
    compare_condition(condition_id, top_n=top_n)

widgets.interact(
    browse_comparison,
    model_a=widgets.Dropdown(options=list(RESULT_FILES), value=COMPARE_MODELS[0], description='Model A'),
    model_b=widgets.Dropdown(options=list(RESULT_FILES), value=COMPARE_MODELS[1], description='Model B'),
    condition_id=widgets.Dropdown(options=ALL_SHARED_CONDITION_IDS, value=ALL_SHARED_CONDITION_IDS[0], description='Condition'),
    top_n=widgets.IntSlider(value=10, min=1, max=39, step=1, description='Rows'),
)

interactive(children=(Dropdown(description='Model A', index=2, options=('MiniLM', 'Decider', 'Qwen3 reranker',…

<function __main__.browse_comparison(model_a, model_b, condition_id, top_n)>

## Across-condition comparison across all method pairs

This section summarizes rank correlation and top-k overlap for signals returned by both methods. JSD and shared weight mass apply only when both methods emit full-candidate distributions; those entries are intentionally blank for graph-reranked results. Comparisons against a shortlist are limited to signals it discovered.

In [12]:
from itertools import combinations

PAIR_CONDITION_METRICS = []
MODEL_PAIRS = list(combinations(RESULT_FILES.keys(), 2))
for model_a, model_b in MODEL_PAIRS:
    condition_ids = sorted(set(RESULTS[model_a]) & set(RESULTS[model_b]))
    for condition_id in condition_ids:
        row_a, row_b = RESULTS[model_a][condition_id], RESULTS[model_b][condition_id]
        ranks_a = _ranked(row_a, model_a)
        ranks_b = _ranked(row_b, model_b)
        rank_a = dict(zip(ranks_a['signal_id'], ranks_a[f'{model_a} rank']))
        rank_b = dict(zip(ranks_b['signal_id'], ranks_b[f'{model_b} rank']))
        ranked_ids = sorted(set(rank_a) & set(rank_b))
        rho = (
            pd.Series([rank_a[key] for key in ranked_ids]).corr(
                pd.Series([rank_b[key] for key in ranked_ids]), method='spearman'
            )
            if len(ranked_ids) >= 2 else math.nan
        )
        top_k_overlap = {}
        for k in (5, 10):
            top_a = set(ranks_a.head(k)['signal_id'])
            top_b = set(ranks_b.head(k)['signal_id'])
            top_k_overlap[k] = len(top_a & top_b)

        if _has_full_distribution(row_a) and _has_full_distribution(row_b):
            weights_a = _normalized_candidate_weights(model_a, row_a)
            weights_b = _normalized_candidate_weights(model_b, row_b)
            common_ids = sorted(set(weights_a) & set(weights_b))
            mass_a = sum(weights_a[key] for key in common_ids)
            mass_b = sum(weights_b[key] for key in common_ids)
            p = {key: weights_a[key] / mass_a for key in common_ids}
            q = {key: weights_b[key] / mass_b for key in common_ids}
            jsd = _jensen_shannon_divergence(p, q)
            shared_mass = sum(min(p[key], q[key]) for key in common_ids)
        else:
            jsd = math.nan
            shared_mass = math.nan

        PAIR_CONDITION_METRICS.append({
            'Model pair': f'{model_a} vs {model_b}',
            'Condition': condition_id,
            'Shared ranked signals': len(ranked_ids),
            'Spearman rho': float(rho) if pd.notna(rho) else math.nan,
            'JSD (bits)': jsd,
            'Shared weight mass': shared_mass,
            'Top-5 overlap': top_k_overlap[5],
            'Top-10 overlap': top_k_overlap[10],
        })

PAIR_METRICS = pd.DataFrame(PAIR_CONDITION_METRICS)
SUMMARY = PAIR_METRICS.groupby('Model pair').agg(
    Conditions=('Condition', 'nunique'),
    Shared_ranked_mean=('Shared ranked signals', 'mean'),
    Spearman_mean=('Spearman rho', 'mean'),
    Spearman_sd=('Spearman rho', 'std'),
    JSD_mean_bits=('JSD (bits)', 'mean'),
    JSD_sd_bits=('JSD (bits)', 'std'),
    Shared_mass_mean=('Shared weight mass', 'mean'),
    Shared_mass_sd=('Shared weight mass', 'std'),
    Top5_overlap_mean=('Top-5 overlap', 'mean'),
    Top10_overlap_mean=('Top-10 overlap', 'mean'),
)
print('Rank similarity uses only signals returned by both methods. JSD/shared-weight metrics are only defined when both methods provide full candidate distributions.')
display(SUMMARY.style.format({
    'Shared_ranked_mean': '{:.1f}', 'Spearman_mean': '{:.3f}', 'Spearman_sd': '{:.3f}',
    'JSD_mean_bits': '{:.3f}', 'JSD_sd_bits': '{:.3f}',
    'Shared_mass_mean': '{:.3f}', 'Shared_mass_sd': '{:.3f}',
    'Top5_overlap_mean': '{:.1f}', 'Top10_overlap_mean': '{:.1f}',
}))
display(PAIR_METRICS.set_index(['Model pair', 'Condition']).style.format({
    'Spearman rho': '{:.3f}', 'JSD (bits)': '{:.3f}',
    'Shared weight mass': '{:.3f}',
}))

pair_labels = list(SUMMARY.index)
colors = plt.cm.tab10.colors
metric_specs = [
    ('Spearman rho', 'Spearman rank correlation', 'Higher = more similar ranks'),
    ('JSD (bits)', 'Jensen–Shannon divergence (bits)', 'Lower = more similar distributions'),
    ('Shared weight mass', 'Shared normalized weight mass', 'Higher = more overlapping weight'),
]
fig, axes = plt.subplots(1, 3, figsize=(17, 4.7))
for ax, (column, title, y_label) in zip(axes, metric_specs):
    condition_label_added = False
    mean_label_added = False
    for index, pair_label in enumerate(pair_labels):
        values = PAIR_METRICS.loc[PAIR_METRICS['Model pair'] == pair_label, column].dropna().tolist()
        offsets = [(j - (len(values) - 1) / 2) * 0.045 for j in range(len(values))]
        if values:
            ax.scatter([index + offset for offset in offsets], values,
                       color=colors[index % len(colors)], alpha=.8, s=42,
                       label='Per condition' if not condition_label_added else None)
            condition_label_added = True
            mean_col = {
                'Spearman rho': 'Spearman_mean', 'JSD (bits)': 'JSD_mean_bits',
                'Shared weight mass': 'Shared_mass_mean',
            }[column]
            sd_col = {
                'Spearman rho': 'Spearman_sd', 'JSD (bits)': 'JSD_sd_bits',
                'Shared weight mass': 'Shared_mass_sd',
            }[column]
            mean, sd = SUMMARY.loc[pair_label, mean_col], SUMMARY.loc[pair_label, sd_col]
            if pd.notna(mean):
                ax.errorbar(index, mean, yerr=sd if pd.notna(sd) else 0,
                            fmt='D', color='black', capsize=4, markersize=5,
                            label='Mean ± SD' if not mean_label_added else None)
                mean_label_added = True
    ax.set(title=title, ylabel=y_label, xticks=range(len(pair_labels)),
           xticklabels=[label.replace(' vs ', '\nvs\n') for label in pair_labels])
    ax.grid(axis='y', alpha=.25)
    if column == 'Spearman rho':
        ax.axhline(0, color='gray', linewidth=.8)
    handles, labels = ax.get_legend_handles_labels()
    if handles:
        ax.legend(frameon=False)
fig.tight_layout()
plt.show()


KeyError: 'Decider graph + history rerank'

## Top-p selection from full-pool rankings or the graph-reranked shortlist

For scoring methods, this selects the shortest ranked prefix whose normalized ranking weights sum to at least top_p. These are relative weights, not calibrated relevance probabilities. Graph-search does not produce a global candidate distribution, so its discovered shortlist is shown in returned order and top-p is not applied.

In [ ]:
CONDITION_ID = 'condition-001'
TOP_P = 0.80

def _inverse_sigmoid(score):
    score = np.clip(float(score), np.finfo(float).tiny, 1 - np.finfo(float).eps)
    return np.log(score) - np.log1p(-score)

def _selection_distribution(model, row):
    if row.get('method') == 'decider-graph-reranked':
        scores = row.get('choice_probabilities') or {}
        ordered = sorted(scores.items(), key=lambda item: (-float(item[1]), item[0]))
        ids = [signal_id for signal_id, _ in ordered]
        probs = np.array([float(score) for _, score in ordered], dtype=float)
        return ids, probs / probs.sum()
    if model == 'Decider':
        scores = row['choice_probabilities']
        ordered = sorted(scores.items(), key=lambda item: (-float(item[1]), item[0]))
        ids = [signal_id for signal_id, _ in ordered]
        probs = np.array([float(score) for _, score in ordered], dtype=float)
        probs = probs / probs.sum()
        return ids, probs
    ordered = sorted(row['signals'], key=lambda s: float(s.get('score', 0)), reverse=True)
    ids = [signal['id'] for signal in ordered]
    scores = np.array([
        float(signal['ranking_logit']) if 'ranking_logit' in signal
        else _inverse_sigmoid(signal['score'])
        for signal in ordered
    ], dtype=float)
    weights = np.exp(scores - scores.max())
    return ids, weights / weights.sum()

def _signal_description(row, signal_id):
    signal = next((s for s in row['signals'] if s['id'] == signal_id), None)
    if signal is None:
        return signal_id, ''
    if signal.get('kind') == 'attribute':
        label = f"{signal.get('attribute_table')}.{signal.get('attribute_column')}"
    else:
        label = 'relationship'
    return label, ' → '.join(signal.get('path', []))

def show_top_p_selection(condition_id=CONDITION_ID, top_p=TOP_P):
    if not 0 < top_p <= 1:
        raise ValueError('top_p must be in (0, 1]')
    for model, model_rows in RESULTS.items():
        row = model_rows[condition_id]
        ids, probs = _selection_distribution(model, row)
        cumulative = np.cumsum(probs)
        count = min(int(np.searchsorted(cumulative, top_p, side='left')) + 1, len(ids))
        records = []
        for rank, (signal_id, probability, cum_prob) in enumerate(zip(ids[:count], probs[:count], cumulative[:count]), 1):
            label, path = _signal_description(row, signal_id)
            records.append({'Rank': rank, 'Signal': label, 'Path': path,
                           'Normalized weight': probability, 'Cumulative weight': cum_prob})
        scope = 'within discovered shortlist' if row.get('method') == 'decider-graph-reranked' else 'over full candidate pool'
        print(f'{model}: {count} signal(s) selected at top_p={top_p:.2f} {scope}; cumulative mass={cumulative[count-1]:.3f}')
        display(pd.DataFrame(records).style.format({'Normalized weight': '{:.4f}', 'Cumulative weight': '{:.4f}'}))

widgets.interact(
    show_top_p_selection,
    condition_id=widgets.Dropdown(options=CONDITION_IDS, value=CONDITION_ID, description='Condition'),
    top_p=widgets.FloatSlider(value=TOP_P, min=0.1, max=1.0, step=0.05, readout_format='.2f', description='Top-p'),
)

## Method results index and browser

This is the compact entry point for all currently available method outputs. Rank-based methods return the full candidate ranking; Decider graph + history-conditioned reranking returns only its discovered shortlist. Its JSONL output also retains the graph-search trace.

Controlled-qrels results are shown in analyze_controlled_signal_results.ipynb. The baseline summary/task files and graph-reranked summary/task files live in /storage/lcostes/ray-share/experiments/simbench-signals/controlled-rel-amazon-v2-ray-20261012/.

In [ ]:
CONTROLLED_RESULT_DIR = Path('/storage/lcostes/ray-share/experiments/simbench-signals/controlled-rel-amazon-v2-ray-20261012')
CONTROLLED_TASKS_BASE = CONTROLLED_RESULT_DIR / 'summary-tasks.jsonl'
RESULT_CATALOG = []
for method, path in RESULT_FILES.items():
    method_rows = RESULTS.get(method, {})
    returned_counts = [len(row.get('signals', [])) for row in method_rows.values()]
    RESULT_CATALOG.append({
        'Method': method,
        'Conditions': len(method_rows),
        'Signals returned (min–max)': (
            f'{min(returned_counts)}–{max(returned_counts)}' if returned_counts else '—'
        ),
        'Output JSONL': str(path),
        'Output exists': path.is_file(),
        'Output type': ('discovered shortlist + history-conditioned ranking' if method == 'Decider graph + history rerank' else 'full candidate ranking'),
        'Controlled-qrels task file': str(CONTROLLED_RESULT_DIR / ('decider-graph-reranked-summary-tasks.jsonl' if method == 'Decider graph + history rerank' else 'summary-tasks.jsonl')),
        'Controlled-qrels file exists': (CONTROLLED_RESULT_DIR / ('decider-graph-reranked-summary-tasks.jsonl' if method == 'Decider graph + history rerank' else 'summary-tasks.jsonl')).is_file(),
    })
display(pd.DataFrame(RESULT_CATALOG).set_index('Method'))

def browse_one_method(method, condition_id, rows_to_show):
    if condition_id not in RESULTS[method]:
        print(f'{condition_id} is not present for {method}.')
        return
    row = RESULTS[method][condition_id]
    ranked = _ranked(row, method).head(rows_to_show)
    signals = {signal['id']: signal for signal in row.get('signals', [])}
    records = []
    for _, result in ranked.iterrows():
        signal = signals[result['signal_id']]
        records.append({
            'rank': int(result[f'{method} rank']),
            'kind': signal.get('kind'),
            'signal': (
                f"{signal.get('attribute_table')}.{signal.get('attribute_column')}"
                if signal.get('kind') == 'attribute' else 'relationship'
            ),
            'path': ' → '.join(signal.get('path') or []),
            'raw score / local probability': (
                signal.get('score') if signal.get('score') is not None
                else (row.get('choice_probabilities') or {}).get(signal['id'])
            ),
        })
    print(row['condition'])
    print(f"Returned {len(row.get('signals', []))} signal(s); method={row.get('method', 'rank')}.")
    display(pd.DataFrame(records))

widgets.interact(
    browse_one_method,
    method=widgets.Dropdown(options=list(RESULTS), value='Decider graph + history rerank', description='Method'),
    condition_id=widgets.Dropdown(options=ALL_SHARED_CONDITION_IDS, value=ALL_SHARED_CONDITION_IDS[0], description='Condition'),
    rows_to_show=widgets.IntSlider(value=10, min=1, max=39, step=1, description='Rows'),
)


## Controlled benchmark comparison: rankers and both graph-root variants

This section compares full-list rankers with graph discovery plus history-conditioned Decider reranking. Model-selected search branches over up to `branch_width` root tables; the controlled-only oracle-root ablation starts from the authored source table. Both can only rank signals discovered by search. Run `analyze_controlled_signal_results.ipynb` for per-condition inspection and the exact/provenance-soft heatmaps.

In [ ]:
CONTROLLED_DIR = Path('/storage/lcostes/ray-share/experiments/simbench-signals/controlled-rel-amazon-v2-ray-20261012')
controlled_files = {
    'Rankers (all)': CONTROLLED_DIR / 'summary.json',
    'Decider graph + history rerank': CONTROLLED_DIR / 'decider-graph-reranked-summary.json',
    'Decider graph + history rerank (oracle root)': CONTROLLED_DIR / 'decider-graph-reranked-oracle-root-summary.json',
}
controlled_summaries = []
for label, path in controlled_files.items():
    if path.is_file():
        payload = json.loads(path.read_text(encoding='utf-8'))
        if label == 'Decider graph + history rerank' and payload.get('search_config', {}).get('root_branch_policy') != 'branch_width_top_actions_at_root':
            print('WARNING: saved model-root graph results predate root branching; rerun this graph evaluator before interpreting them.')
        controlled_summaries.append((label, payload))
    else:
        print(f'Missing {label} results: {path}')
if controlled_summaries:
    controlled_rows = []
    method_order = ['random', 'minilm', 'qwen3-reranker', 'decider', 'decider-graph', 'decider-graph-oracle-root']
    for label, payload in controlled_summaries:
        for scorer, metrics in payload['scorers'].items():
            method_label = scorer if label == 'Rankers (all)' else label
            method_label = {'decider-graph-reranked': 'decider-graph', 'decider-graph-reranked-oracle-root': 'decider-graph-oracle-root', 'Decider graph + history rerank': 'decider-graph', 'Decider graph + history rerank (oracle root)': 'decider-graph-oracle-root'}.get(method_label, method_label)
            controlled_rows.append({'method': method_label, **metrics})
    controlled_comparison = pd.DataFrame(controlled_rows).drop_duplicates('method', keep='last')
    controlled_comparison['method'] = pd.Categorical(controlled_comparison['method'], categories=method_order, ordered=True)
    controlled_comparison = controlled_comparison.sort_values('method')
    display(controlled_comparison[['method', 'task_count', 'ndcg@3', 'recall@3', 'mrr']].style.format({'ndcg@3': '{:.3f}', 'recall@3': '{:.3f}', 'mrr': '{:.3f}'}))
    # Oracle-root is shown as a hatched delta on each decider-graph metric bar, not as a separate method.
    from matplotlib.patches import Patch
    methods = [str(value) for value in controlled_comparison['method'].dropna().tolist() if str(value) != 'decider-graph-oracle-root']
    oracle_row = controlled_comparison.loc[controlled_comparison['method'].astype(str) == 'decider-graph-oracle-root']
    oracle_values = oracle_row.iloc[0] if not oracle_row.empty else None
    metrics = ['ndcg@3', 'recall@3', 'mrr']
    positions = list(range(len(methods)))
    width = 0.8 / len(metrics)
    colors = plt.rcParams['axes.prop_cycle'].by_key()['color']
    fig, ax = plt.subplots(figsize=(11, 4.5))
    for metric_index, metric in enumerate(metrics):
        values = [float(controlled_comparison.loc[controlled_comparison['method'].astype(str) == method, metric].iloc[0]) for method in methods]
        offset = (metric_index - (len(metrics) - 1) / 2) * width
        color = colors[metric_index % len(colors)]
        ax.bar([position + offset for position in positions], values, width, label=metric, color=color)
        if 'decider-graph' in methods and oracle_values is not None:
            graph_index = methods.index('decider-graph')
            base = values[graph_index]
            oracle = float(oracle_values[metric])
            ax.bar(positions[graph_index] + offset, abs(oracle - base), width, bottom=min(base, oracle), facecolor='none', edgecolor=color, hatch='....', linewidth=1.2, zorder=4)
    ax.set_xticks(positions, methods)
    ax.set_ylim(0, 1.05)
    ax.set_ylabel('Macro average over 25 synthetic conditions')
    ax.set_title('Full-list methods versus history-conditioned graph reranking')
    ax.grid(axis='y', alpha=0.25)
    plt.xticks(rotation=25, ha='right')
    handles, labels = ax.get_legend_handles_labels()
    if oracle_values is not None:
        handles.append(Patch(facecolor='white', edgecolor='0.25', hatch='....', label='Oracle-root delta'))
        labels.append('Oracle-root delta')
    ax.legend(handles, labels, title='Metric', bbox_to_anchor=(1.02, 1), loc='upper left')
    plt.tight_layout()
    plt.show()

    for task_filename, description in [(
        'decider-graph-reranked-summary-tasks.jsonl', 'model-root graph rerank'),
        ('decider-graph-reranked-oracle-root-summary-tasks.jsonl', 'oracle-root graph rerank'),
    ]:
        task_path = CONTROLLED_DIR / task_filename
        if task_path.is_file():
            rows = [json.loads(line) for line in task_path.read_text(encoding='utf-8').splitlines() if line.strip()]
            print(f'Loaded {len(rows)} {description} per-condition results from {task_path}')
            if description == 'model-root graph rerank' and rows and 'root_selection' in rows[0]:
                root_correct = sum(bool(row['root_selection']['correct']) for row in rows)
                root_hit = sum(bool(row['root_selection'].get('explored_correct', row['root_selection']['correct'])) for row in rows)
                print(f'Model-selected top-1 root accuracy: {root_correct}/{len(rows)} = {root_correct / len(rows):.1%}; source table explored: {root_hit}/{len(rows)} = {root_hit / len(rows):.1%}')
        else:
            print(f'Missing {description} per-condition results: {task_path}')